# CNMP — Modelo semântico (Resolução 277)

Cria (ou recria) o modelo semântico Direct Lake sobre o Warehouse `mp_gold`
por código, com `semantic-link-labs`: descobre as tabelas largas e suas
partes (`fato_visita_{id}`, `_p2`, `_p3`, ...), cria os relacionamentos,
oculta as chaves técnicas e adiciona as medidas iniciais.

As decisões de modelagem estão documentadas em `03_modelo_semantico.md`.

**Pré-requisitos:**
- Rodar `02_carga_gold.ipynb` antes, com o pacote atualizado (aliases e
  `SELECT DISTINCT` em `dim_unidade`); sem isso os relacionamentos com
  `dim_unidade` falham por chave duplicada.
- A capacidade do workspace precisa ter o endpoint XMLA em leitura e escrita
  (Admin portal > Capacity settings > XMLA endpoint = Read Write), que é como
  o TOM aplica relacionamentos e medidas.
- A identidade do notebook precisa poder criar itens no workspace.

In [ ]:
%pip install --quiet semantic-link-labs git+https://github.com/mpsp-jurimetria/proj202607.git#subdirectory=python

## Configuração

Mesma observação dos outros notebooks: não são segredos, mas evite deixar
valores reais commitados aqui.

In [ ]:
import os

os.environ["FABRIC_WAREHOUSE_GOLD_HOST"] = "<host>.datawarehouse.fabric.microsoft.com"
os.environ["FABRIC_WAREHOUSE_GOLD_NAME"] = "mp_gold"

WORKSPACE = "<nome do workspace>"
WAREHOUSE = "mp_gold"
DATASET = "mp_gold_resolucao_277"

# Dimensões incluídas no modelo; dim_campo, dim_campo_opcao, dim_campo_alias e
# fato_resposta_tipada ficam fora (uso ad hoc, ver 03_modelo_semantico.md).
DIMENSOES = ["dim_unidade", "dim_formulario"]

# Chaves técnicas, ocultadas em todas as tabelas do modelo.
CHAVES_TECNICAS = {"instancia_id_api", "entidade_id_api", "formulario_id_api", "ambiente_id_api"}

## Descoberta das tabelas do gold

Lê o `INFORMATION_SCHEMA` do warehouse para achar as tabelas largas e suas
partes; o modelo é recriado do zero a cada execução, então formulários novos
ou repartições diferentes entram sozinhos.

In [ ]:
import re

from sqlalchemy import text

from src.infra.warehouse import get_gold_engine

engine = get_gold_engine()
with engine.connect() as conn:
    tabelas_gold = [
        linha[0]
        for linha in conn.execute(text(
            "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES "
            "WHERE TABLE_SCHEMA = 'dbo' AND TABLE_TYPE = 'BASE TABLE'"
        ))
    ]

padrao_larga = re.compile(r"^fato_visita_(\d+)(?:_p(\d+))?$")
principais: list[str] = []
partes: list[str] = []
for tabela in tabelas_gold:
    m = padrao_larga.match(tabela)
    if m:
        (partes if m.group(2) else principais).append(tabela)

chave_natural = lambda t: [int(x) for x in re.findall(r"\d+", t)]
principais.sort(key=chave_natural)
partes.sort(key=chave_natural)

TABELAS_MODELO = DIMENSOES + ["fato_visita"] + principais + partes
print(f"{len(principais)} tabelas largas principais: {principais}")
print(f"{len(partes)} partes: {partes}")
print(f"{len(TABELAS_MODELO)} tabelas no modelo")

## Criação do modelo Direct Lake

`overwrite=True` recria o modelo do zero; os relacionamentos e medidas são
reaplicados na célula seguinte, então rodar o notebook inteiro é idempotente.
Atenção: personalizações feitas à mão no modelo são perdidas na recriação —
se fizer ajuste manual que valha a pena manter, traga-o para este notebook.

In [ ]:
from sempy_labs.directlake import generate_direct_lake_semantic_model

generate_direct_lake_semantic_model(
    dataset=DATASET,
    tables=TABELAS_MODELO,
    source=WAREHOUSE,
    source_type="Warehouse",
    workspace=WORKSPACE,
    overwrite=True,
)

## Relacionamentos, chaves ocultas e medidas

- Fatos -> dimensões: muitos-para-um, filtro em direção única.
- Cada parte -> tabela principal do formulário: um-para-um em
  `instancia_id_api`, filtro nas duas direções (para a segmentação por
  unidade/ano alcançar as colunas das partes).
- Medidas de capacidade/ocupação do 1322 são geradas somando as colunas com
  prefixo `q3_1_`/`q3_2_` (seção III, ambos os sexos), onde quer que elas
  tenham caído na divisão em partes.

In [ ]:
from sempy_labs.tom import connect_semantic_model


def _tabela_principal(parte: str) -> str:
    return re.sub(r"_p\d+$", "", parte)


def _soma_por_prefixo(tom, familia: str, prefixo: str) -> str:
    """Expressão DAX somando todas as colunas da família de tabelas de um
    formulário (principal + partes) cujo nome começa com o prefixo."""
    termos = [
        f"SUM('{tabela.Name}'[{coluna.Name}])"
        for tabela in tom.model.Tables
        if tabela.Name == familia or tabela.Name.startswith(f"{familia}_p")
        for coluna in tabela.Columns
        if coluna.Name.startswith(prefixo)
    ]
    return " + ".join(termos)


with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    tom.add_relationship(
        from_table="fato_visita", from_column="entidade_id_api",
        to_table="dim_unidade", to_column="entidade_id_api",
        from_cardinality="Many", to_cardinality="One",
    )
    tom.add_relationship(
        from_table="fato_visita", from_column="formulario_id_api",
        to_table="dim_formulario", to_column="formulario_id_api",
        from_cardinality="Many", to_cardinality="One",
    )
    for principal in principais:
        tom.add_relationship(
            from_table=principal, from_column="entidade_id_api",
            to_table="dim_unidade", to_column="entidade_id_api",
            from_cardinality="Many", to_cardinality="One",
        )
    for parte in partes:
        tom.add_relationship(
            from_table=parte, from_column="instancia_id_api",
            to_table=_tabela_principal(parte), to_column="instancia_id_api",
            from_cardinality="One", to_cardinality="One",
            cross_filtering_behavior="BothDirections",
        )

    for tabela in tom.model.Tables:
        for coluna in tabela.Columns:
            if coluna.Name in CHAVES_TECNICAS:
                coluna.IsHidden = True

    tom.add_measure(
        table_name="fato_visita", measure_name="Visitas",
        expression="COUNTROWS(fato_visita)", format_string="#,0",
    )
    tom.add_measure(
        table_name="fato_visita", measure_name="Unidades visitadas",
        expression="DISTINCTCOUNT(fato_visita[entidade_id_api])", format_string="#,0",
    )

    if "fato_visita_1322" in principais:
        capacidade = _soma_por_prefixo(tom, "fato_visita_1322", "q3_1_")
        ocupacao = _soma_por_prefixo(tom, "fato_visita_1322", "q3_2_")
        if capacidade and ocupacao:
            tom.add_measure(
                table_name="fato_visita_1322", measure_name="Capacidade total (1322)",
                expression=capacidade, format_string="#,0",
            )
            tom.add_measure(
                table_name="fato_visita_1322", measure_name="Ocupação total (1322)",
                expression=ocupacao, format_string="#,0",
            )
            tom.add_measure(
                table_name="fato_visita_1322", measure_name="Taxa de ocupação (1322)",
                expression="DIVIDE([Ocupação total (1322)], [Capacidade total (1322)])",
                format_string="0.0%",
            )

print("Relacionamentos, chaves ocultas e medidas aplicados")

## Verificação

In [ ]:
import sempy_labs as labs

relacionamentos = labs.list_relationships(dataset=DATASET, workspace=WORKSPACE)
print(f"{len(relacionamentos)} relacionamentos")
display(relacionamentos)